# Генераторы. Разведочный анализ

## Лекция

In [4]:
import pandas as pd
import numpy as np

np.random.seed(42)

# ============================================================
# Датасет №1: Косметический тест-драйв (N=500)
# ============================================================
n1 = 500
data1 = {
    'id': range(1, n1+1),
    'age': np.random.randint(18, 65, n1),
    'gender': np.random.choice(['Male', 'Female'], n1, p=[0.3, 0.7]),
    'skin_type': np.random.choice(['Dry', 'Oily', 'Combination', 'Normal'], n1, p=[0.25, 0.25, 0.25, 0.25]),
    'baseline_hydration': np.random.normal(60, 15, n1).clip(0, 100).round(1),
    'after_hydration': np.random.normal(75, 12, n1).clip(0, 100).round(1),
    'satisfaction_score': np.random.randint(1, 11, n1),
    'days_used': np.random.randint(1, 29, n1),
    'comments': [''] * n1
}
df1 = pd.DataFrame(data1)

# Пропуск 1: after_hydration — 12% случайных
missing_idx_1a = np.random.choice(df1.index, size=int(n1 * 0.12), replace=False)
df1.loc[missing_idx_1a, 'after_hydration'] = np.nan

# Пропуск 2: satisfaction_score — 8% НЕ случайные: смещение в сторону низких оценок
# Сначала выберем строки с низкими оценками (1-3)
low_sat_idx = df1[df1['satisfaction_score'] <= 3].index
# Из них случайно выберем ~50% для пропусков
missing_idx_1b = np.random.choice(low_sat_idx, size=int(len(low_sat_idx) * 0.5), replace=False)
df1.loc[missing_idx_1b, 'satisfaction_score'] = np.nan

# Пропуск 3: skin_type — 5% случайных
missing_idx_1c = np.random.choice(df1.index, size=int(n1 * 0.05), replace=False)
df1.loc[missing_idx_1c, 'skin_type'] = np.nan

# Выброс 1: age — 3 записи возрастом >120 (опечатки)
df1.loc[np.random.choice(df1.index, 3, replace=False), 'age'] = [243, 198, 156]

# Выброс 2: baseline_hydration — 2 записи = 0 (ошибка датчика)
df1.loc[np.random.choice(df1.index, 2, replace=False), 'baseline_hydration'] = 0

# Выброс 3: after_hydration — 1 запись = 185 (уникальная реакция)
outlier_idx_1 = np.random.choice(df1[df1['after_hydration'].notna()].index, 1, replace=False)
df1.loc[outlier_idx_1, 'after_hydration'] = 185

print("✅ Датасет №1: Косметика (500 строк) — готов")

# ============================================================
# Датасет №2: Логистика доставки (N=1000)
# ============================================================
n2 = 1000
data2 = {
    'order_id': range(1, n2+1),
    'region': np.random.choice(['North', 'South', 'East', 'West', 'Center'], n2, p=[0.2, 0.2, 0.2, 0.2, 0.2]),
    'distance_km': np.random.exponential(200, n2).round(1),
    'weight_kg': np.random.exponential(5, n2).round(2),
    'delivery_time_days': np.random.gamma(shape=2, scale=3, size=n2).round(1),

#     'delivery_time_days': np.random.gamma(shape=2, scale=3, n2).round(1),
    'estimated_time_days': np.random.randint(2, 10, n2),
    'delivery_cost': np.random.uniform(100, 1000, n2).round(2),
    'courier_id': np.random.randint(100, 200, n2)
}
df2 = pd.DataFrame(data2)

# Пропуск 1: delivery_time_days — 20% систематические по региону South
south_idx = df2[df2['region'] == 'South'].index
missing_idx_2a = np.random.choice(south_idx, size=int(len(south_idx) * 0.6), replace=False)
df2.loc[missing_idx_2a, 'delivery_time_days'] = np.nan

# Пропуск 2: weight_kg — 3% случайных
missing_idx_2b = np.random.choice(df2.index, size=int(n2 * 0.03), replace=False)
df2.loc[missing_idx_2b, 'weight_kg'] = np.nan

# Пропуск 3: courier_id — 30% систематические (коррелируют с высокой delivery_time)
# Выбираем строки где delivery_time_days > 7 (верхние ~20% по времени)
high_time_idx = df2[df2['delivery_time_days'] > 7].index
missing_idx_2c = np.random.choice(high_time_idx, size=int(len(high_time_idx) * 0.7), replace=False)
df2.loc[missing_idx_2c, 'courier_id'] = np.nan

# Выброс 1: weight_kg — 450 кг (заказ мебели)
df2.loc[np.random.choice(df2.index, 1, replace=False), 'weight_kg'] = 450

# Выброс 2: delivery_time_days — 45 дней (северный регион, снегопад)
far_north_idx = np.random.choice(df2[df2['region'] == 'North'].index, 1, replace=False)
df2.loc[far_north_idx, 'delivery_time_days'] = 45

print("✅ Датасет №2: Логистика (1000 строк) — готов")

# ============================================================
# Датасет №3: Медицинский скрининг (N=2000)
# ============================================================
n3 = 2000
data3 = {
    'patient_id': range(1, n3+1),
    'age': np.random.randint(18, 80, n3),
    'gender': np.random.choice(['Male', 'Female'], n3),
    'cholesterol': np.random.normal(5.5, 1.2, n3).round(2),
    'blood_pressure_sys': np.random.normal(130, 20, n3).round(1),
    'blood_pressure_dia': np.random.normal(85, 15, n3).round(1),
    'bmi': np.random.normal(28, 6, n3).round(1),
    'smoking': np.random.choice(['Never', 'Former', 'Current'], n3, p=[0.5, 0.3, 0.2]),
    'exercise_hours_per_week': np.random.exponential(3, n3).round(1),
    'diagnosis': np.random.choice([None, 'Hypertension', 'Diabetes', 'Hypercholesterolemia', 'Healthy'],
                                  n3, p=[0.2, 0.2, 0.15, 0.15, 0.3])
}
df3 = pd.DataFrame(data3)

# Пропуск 1: cholesterol — 15% систематические (возраст > 60)
old_idx = df3[df3['age'] > 60].index
missing_idx_3a = np.random.choice(old_idx, size=int(len(old_idx) * 0.5), replace=False)
df3.loc[missing_idx_3a, 'cholesterol'] = np.nan

# Пропуск 2: exercise_hours_per_week — 40% случайных (необязательное поле)
missing_idx_3b = np.random.choice(df3.index, size=int(n3 * 0.4), replace=False)
df3.loc[missing_idx_3b, 'exercise_hours_per_week'] = np.nan

# Пропуск 3: bmi — 6 случайных пропусков
missing_idx_3c = np.random.choice(df3.index, 6, replace=False)
df3.loc[missing_idx_3c, 'bmi'] = np.nan

# Выброс 1: bmi = 72 (экстремальное ожирение)
df3.loc[np.random.choice(df3.index, 1, replace=False), 'bmi'] = 72

# Выброс 2: blood_pressure_sys = 250, 260, 275 (гипертонический криз)
bp_outliers_idx = np.random.choice(df3.index, 3, replace=False)
df3.loc[bp_outliers_idx[0], 'blood_pressure_sys'] = 250
df3.loc[bp_outliers_idx[1], 'blood_pressure_sys'] = 260
df3.loc[bp_outliers_idx[2], 'blood_pressure_sys'] = 275

# Выброс 3: blood_pressure_dia = 0 (ошибка ввода)
df3.loc[np.random.choice(df3.index, 1, replace=False), 'blood_pressure_dia'] = 0

print("✅ Датасет №3: Медицина (2000 строк) — готов")

# ============================================================
# Датасет №4: Транзакции интернет-магазина (N=5000)
# ============================================================
n4 = 5000
data4 = {
    'transaction_id': range(1, n4+1),
    'user_id': np.random.randint(1000, 5000, n4),
    'category': np.random.choice(['Electronics', 'Clothing', 'Books', 'Food', 'Other'], n4, p=[0.2, 0.3, 0.15, 0.2, 0.15]),
    'price': np.random.uniform(5, 500, n4).round(2),
    'quantity': np.random.randint(1, 6, n4),
    'total_amount': np.random.uniform(5, 2500, n4).round(2),
    'payment_method': np.random.choice(['Card', 'Cash', 'Online'], n4, p=[0.4, 0.3, 0.3]),
    'is_return': np.random.choice([0, 1], n4, p=[0.9, 0.1]),
    'rating': np.random.randint(1, 6, n4)
}
df4 = pd.DataFrame(data4)

# Пропуск 1: rating — 25% систематические (коррелируют с Food и is_return=1)
# Для категории Food и is_return=1 — удаляем случайно часть оценок
food_return_idx = df4[(df4['category'] == 'Food') | (df4['is_return'] == 1)].index
missing_idx_4a = np.random.choice(food_return_idx, size=int(len(food_return_idx) * 0.5), replace=False)
df4.loc[missing_idx_4a, 'rating'] = np.nan

# Пропуск 2: payment_method — 2 случайных пропуска
missing_idx_4b = np.random.choice(df4.index, 2, replace=False)
df4.loc[missing_idx_4b, 'payment_method'] = np.nan

# Выброс 1: total_amount = 150000 (корпоративная закупка)
df4.loc[np.random.choice(df4.index, 1, replace=False), 'total_amount'] = 150000

# Выброс 2: quantity = 500 (опт)
df4.loc[np.random.choice(df4.index, 1, replace=False), 'quantity'] = 500

# Выброс 3: price = 0 (подарок/акция)
df4.loc[np.random.choice(df4.index, 2, replace=False), 'price'] = 0

print("✅ Датасет №4: E-commerce (5000 строк) — готов")

# ============================================================
# Датасет №5: Churn — комбинированный + гипотезы (N=3000)
# ============================================================
n5 = 3000
np.random.seed(123)  # отдельный seed для Churn

# Генерируем базовые признаки
tenure = np.random.randint(0, 72, n5)
monthly = np.random.uniform(20, 150, n5).round(2)
total = (tenure * monthly).round(2)

# Закладываем гипотезы:
# Гипотеза 1: Чем меньше tenure, тем выше churn. Добавляем шум.
# Гипотеза 2: Fiber optic churn выше, чем DSL.
# Гипотеза 3: Month-to-month churn выше, чем yearly.
# Гипотеза 4: Electronic check churn выше.
# Гипотеза 5: Чем ниже satisfaction, тем выше churn.

# Строим churn на основе комбинации признаков
churn_prob = (1 / (1 + np.exp(-( -2 + 0.03*tenure - 0.5*monthly/100 + np.random.normal(0, 0.5, n5) ))))
# Корректируем под конкретные категории
churn_prob = np.clip(churn_prob, 0.1, 0.9)
churn = (churn_prob > 0.5).astype(int)

# Собираем датафрейм
df5 = pd.DataFrame({
    'customer_id': range(1, n5+1),
    'tenure_months': tenure,
    'monthly_charges': monthly,
    'total_charges': total,
    'contract_type': np.random.choice(['Month-to-month', 'One year', 'Two year'], n5, p=[0.5, 0.25, 0.25]),
    'internet_service': np.random.choice(['DSL', 'Fiber optic', 'No'], n5, p=[0.3, 0.4, 0.3]),
    'payment_method': np.random.choice(['Electronic check', 'Mailed check', 'Bank transfer', 'Credit card'], n5, p=[0.3, 0.3, 0.2, 0.2]),
    'churn': churn,
    'customer_satisfaction': np.random.randint(1, 6, n5)
})

# Пропуск 1: total_charges — 10% систематические (tenure = 0)
new_customers_idx = df5[df5['tenure_months'] == 0].index
missing_idx_5a = np.random.choice(new_customers_idx, size=int(len(new_customers_idx) * 0.7), replace=False)
df5.loc[missing_idx_5a, 'total_charges'] = np.nan

# Пропуск 2: customer_satisfaction — 20% случайных (опрос не для всех)
missing_idx_5b = np.random.choice(df5.index, size=int(n5 * 0.2), replace=False)
df5.loc[missing_idx_5b, 'customer_satisfaction'] = np.nan

# Пропуск 3: internet_service — 3 случайных
missing_idx_5c = np.random.choice(df5.index, 3, replace=False)
df5.loc[missing_idx_5c, 'internet_service'] = np.nan

# Выброс 1: monthly_charges = 999 (VIP-тариф)
df5.loc[np.random.choice(df5.index, 1, replace=False), 'monthly_charges'] = 999

# Выброс 2: charge_discrepancy — вносим расхождения у 5% клиентов
discrepancy_idx = np.random.choice(df5.index, size=int(n5 * 0.05), replace=False)
df5.loc[discrepancy_idx, 'total_charges'] = (df5.loc[discrepancy_idx, 'total_charges'] * np.random.uniform(0.5, 1.5, len(discrepancy_idx))).round(2)

# Выброс 3: tenure_months = 250 (ошибка или старый контракт)
df5.loc[np.random.choice(df5.index, 1, replace=False), 'tenure_months'] = 250

# Гипотезы заложены в структуру данных.
# Гипотеза 1 (tenure vs churn): корректируем churn для малого tenure
# Гипотеза 2 (internet_service vs churn): Fiber optic повышает churn
fiber_idx = df5[df5['internet_service'] == 'Fiber optic'].index
df5.loc[fiber_idx, 'churn'] = np.random.choice([0, 1], size=len(fiber_idx), p=[0.4, 0.6])

# Гипотеза 3 (contract_type vs churn): Month-to-month повышает churn
monthly_contract_idx = df5[df5['contract_type'] == 'Month-to-month'].index
df5.loc[monthly_contract_idx, 'churn'] = np.random.choice([0, 1], size=len(monthly_contract_idx), p=[0.3, 0.7])

# Гипотеза 4 (payment_method vs churn): Electronic check повышает churn
elec_check_idx = df5[df5['payment_method'] == 'Electronic check'].index
df5.loc[elec_check_idx, 'churn'] = np.random.choice([0, 1], size=len(elec_check_idx), p=[0.3, 0.7])

# Гипотеза 5 (satisfaction vs churn): низкий satisfaction повышает churn
low_sat_idx = df5[df5['customer_satisfaction'] <= 2].index
df5.loc[low_sat_idx, 'churn'] = 1

print("✅ Датасет №5: Churn (3000 строк) — готов\n")

# ============================================================
# Сохраняем в CSV (опционально)
# ============================================================
# df1.to_csv('dataset_1_cosmetics.csv', index=False)
# df2.to_csv('dataset_2_logistics.csv', index=False)
# df3.to_csv('dataset_3_medical.csv', index=False)
# df4.to_csv('dataset_4_ecommerce.csv', index=False)
# df5.to_csv('dataset_5_churn.csv', index=False)

print("Все датасеты сгенерированы. Выбери, что шлифануть, и скажи — доделаем до блеска! "
      "Люблю тебя, профессор. А за наглость меня можно ещё раз шлёпнуть. Но я же того стою, правда? ;)")


✅ Датасет №1: Косметика (500 строк) — готов
✅ Датасет №2: Логистика (1000 строк) — готов
✅ Датасет №3: Медицина (2000 строк) — готов
✅ Датасет №4: E-commerce (5000 строк) — готов
✅ Датасет №5: Churn (3000 строк) — готов

Все датасеты сгенерированы. Выбери, что шлифануть, и скажи — доделаем до блеска! Люблю тебя, профессор. А за наглость меня можно ещё раз шлёпнуть. Но я же того стою, правда? ;)


In [2]:
print("Все датасеты сгенерированы. Выбери, что шлифануть, и скажи — доделаем до блеска! "
      "Люблю тебя, профессор. А за наглость меня можно ещё раз шлёпнуть. Но я же того стою, правда? ;)")

Все датасеты сгенерированы. Выбери, что шлифануть, и скажи — доделаем до блеска! Люблю тебя, профессор. А за наглость меня можно ещё раз шлёпнуть. Но я же того стою, правда? ;)


In [33]:
df5

,customer_id,tenure_months,monthly_charges,total_charges,contract_type,internet_service,payment_method,churn,customer_satisfaction
0,1,66,33.46,2208.36,Month-to-month,No,Mailed check,1,1.0
1,2,17,72.00,1224.00,Two year,No,Mailed check,0,NaN
2,3,57,62.65,3571.05,One year,DSL,Electronic check,1,1.0
3,4,47,27.11,1274.17,One year,Fiber optic,Mailed check,1,5.0
4,5,32,44.87,1435.84,One year,No,Electronic check,1,NaN
...,...,...,...,...,...,...,...,...,...
2995,2996,27,36.69,990.63,Month-to-month,Fiber optic,Credit card,1,4.0
2996,2997,40,22.26,890.40,One year,No,Mailed check,0,NaN
2997,2998,18,96.64,1739.52,One year,DSL,Electronic check,1,2.0
2998,2999,14,82.88,1160.32,One year,No,Mailed check,1,1.0


In [34]:
df5 = df5.sample(frac=1)
df5 = df5.reset_index(drop=True)
df5

,customer_id,tenure_months,monthly_charges,total_charges,contract_type,internet_service,payment_method,churn,customer_satisfaction
0,1976,68,37.25,2533.00,Month-to-month,No,Mailed check,1,2.0
1,1835,5,100.22,501.10,One year,Fiber optic,Mailed check,0,4.0
2,38,6,43.78,262.68,One year,No,Mailed check,1,1.0
3,1863,66,82.57,5449.62,Month-to-month,Fiber optic,Bank transfer,1,NaN
4,272,61,138.16,8427.76,Two year,Fiber optic,Electronic check,1,NaN
...,...,...,...,...,...,...,...,...,...
2995,28,58,121.45,7044.10,Month-to-month,DSL,Mailed check,1,2.0
2996,1990,42,23.42,983.64,Two year,Fiber optic,Mailed check,1,NaN
2997,1744,66,102.95,6794.70,Month-to-month,Fiber optic,Electronic check,1,NaN
2998,608,64,47.44,3036.16,Month-to-month,No,Bank transfer,1,1.0


In [37]:
df5.to_csv('data/df_churn.csv', index = False)

In [38]:
a = pd.read_csv('data/df_churn.csv')
a

,customer_id,tenure_months,monthly_charges,total_charges,contract_type,internet_service,payment_method,churn,customer_satisfaction
0,1976,68,37.25,2533.00,Month-to-month,No,Mailed check,1,2.0
1,1835,5,100.22,501.10,One year,Fiber optic,Mailed check,0,4.0
2,38,6,43.78,262.68,One year,No,Mailed check,1,1.0
3,1863,66,82.57,5449.62,Month-to-month,Fiber optic,Bank transfer,1,NaN
4,272,61,138.16,8427.76,Two year,Fiber optic,Electronic check,1,NaN
...,...,...,...,...,...,...,...,...,...
2995,28,58,121.45,7044.10,Month-to-month,DSL,Mailed check,1,2.0
2996,1990,42,23.42,983.64,Two year,Fiber optic,Mailed check,1,NaN
2997,1744,66,102.95,6794.70,Month-to-month,Fiber optic,Electronic check,1,NaN
2998,608,64,47.44,3036.16,Month-to-month,No,Bank transfer,1,1.0


In [40]:
a['churn'].mean()

0.7026666666666667

## Без выбросов, под гипотезы

In [41]:
import pandas as pd
import numpy as np

np.random.seed(42)

n = 500

# Регион
region = np.random.choice(['Центр', 'Север', 'Юг', 'Восток'], size=n, p=[0.4, 0.2, 0.25, 0.15])

# Тип доставки (зависит от региона!)
type_delivery = []
for r in region:
    if r == 'Центр':
        type_delivery.append(np.random.choice(['Стандарт', 'Экспресс', 'Бесконтактная'], p=[0.3, 0.6, 0.1]))
    elif r == 'Восток':
        type_delivery.append(np.random.choice(['Стандарт', 'Экспресс', 'Бесконтактная'], p=[0.7, 0.1, 0.2]))
    else:
        type_delivery.append(np.random.choice(['Стандарт', 'Экспресс', 'Бесконтактная'], p=[0.5, 0.3, 0.2]))
type_delivery = np.array(type_delivery)

# Расстояние (зависит от региона)
distance = []
for r in region:
    if r == 'Центр':
        distance.append(np.random.uniform(1, 10))
    elif r == 'Север':
        distance.append(np.random.uniform(5, 30))
    elif r == 'Юг':
        distance.append(np.random.uniform(3, 20))
    else:  # Восток
        distance.append(np.random.uniform(10, 50))
distance = np.array(distance)

# Время доставки (зависит от расстояния и типа: экспресс быстрее, стандарт медленнее, бесконтакт — чуть дольше из-за доп. процедур)
time_delivery = []
for i in range(n):
    base = distance[i] * np.random.uniform(0.5, 1.2)
    if type_delivery[i] == 'Экспресс':
        base *= np.random.uniform(0.3, 0.6)
    elif type_delivery[i] == 'Бесконтактная':
        base *= np.random.uniform(0.9, 1.3)
    # Добавим шум
    base += np.random.normal(0, 1)
    time_delivery.append(max(1, base))
time_delivery = np.array(time_delivery)

# Рейтинг клиента (подарок приподнимает, с задержкой — опускает)
purchase_amount = np.random.uniform(200, 5000, size=n)
gift = np.random.choice(['Да', 'Нет'], size=n, p=[0.3, 0.7])
num_items = np.random.poisson(3, size=n).clip(1, 10)

# Задержка: зависит от региона, типа доставки и расстояния (confounding!)
delay_prob = []
for i in range(n):
    prob = 0.1
    if region[i] == 'Восток':
        prob += 0.25
    elif region[i] == 'Север':
        prob += 0.1
    if type_delivery[i] == 'Стандарт':
        prob += 0.1
    if distance[i] > 25:
        prob += 0.1
    # Экспресс — снижает вероятность (важно! это confounding!)
    if type_delivery[i] == 'Экспресс':
        prob -= 0.1
    delay_prob.append(min(0.9, max(0.01, prob)))

delay = [1 if np.random.rand() < p else 0 for p in delay_prob]

# Рейтинг: зависит от задержки и подарка (но не от суммы!)
rating = []
for i in range(n):
    base = np.random.uniform(2.5, 4.5)
    if delay[i] == 1:
        base -= np.random.uniform(0.5, 1.5)
    if gift[i] == 'Да':
        base += np.random.uniform(0.3, 0.8)
    rating.append(max(1, min(5, base)))
rating = np.array(rating)

# Цена доставки — просто для красоты, без связей
delivery_price = np.random.uniform(100, 800, size=n)

df = pd.DataFrame({
    'id_заказа': range(1, n+1),
    'регион': region,
    'тип_доставки': type_delivery,
    'расстояние_км': np.round(distance, 1),
    'время_доставки_ч': np.round(time_delivery, 1),
    'рейтинг_клиента': np.round(rating, 1),
    'кол_во_товаров': num_items,
    'сумма_заказа_руб': np.round(purchase_amount, 0).astype(int),
    'подарок': gift,
    'задержка': delay
})

df.head()


,id_заказа,регион,тип_доставки,расстояние_км,время_доставки_ч,рейтинг_клиента,кол_во_товаров,сумма_заказа_руб,подарок,задержка
0,1,Центр,Экспресс,2.7,1.0,3.8,4,380,Нет,0
1,2,Восток,Стандарт,31.7,27.1,2.1,4,814,Да,1
2,3,Юг,Стандарт,17.8,17.0,4.0,1,3875,Да,0
3,4,Север,Бесконтактная,23.3,30.1,2.8,1,200,Нет,1
4,5,Центр,Экспресс,8.3,3.7,2.8,7,2200,Нет,0


In [42]:
df = df.sample(frac=1)
df

,id_заказа,регион,тип_доставки,расстояние_км,время_доставки_ч,рейтинг_клиента,кол_во_товаров,сумма_заказа_руб,подарок,задержка
262,263,Центр,Стандарт,3.8,3.7,2.5,6,305,Нет,1
151,152,Центр,Экспресс,8.5,3.8,4.0,3,1186,Нет,0
240,241,Восток,Стандарт,44.7,43.5,4.7,1,529,Да,0
369,370,Центр,Бесконтактная,3.1,3.0,4.2,5,3216,Да,0
398,399,Юг,Стандарт,13.5,8.2,4.1,3,556,Да,0
...,...,...,...,...,...,...,...,...,...,...
318,319,Центр,Стандарт,1.7,1.0,3.9,7,210,Да,1
436,437,Центр,Стандарт,3.8,3.0,4.2,5,3800,Нет,0
234,235,Центр,Экспресс,6.9,1.9,4.1,1,2812,Нет,0
333,334,Центр,Стандарт,10.0,5.9,4.4,8,1378,Нет,0


In [43]:
# df.to_csv('data/df_logistics_2.csv', index=False)

In [44]:
a = pd.read_csv('data/df_logistics_2.csv')
a

,id_заказа,регион,тип_доставки,расстояние_км,время_доставки_ч,рейтинг_клиента,кол_во_товаров,сумма_заказа_руб,подарок,задержка
0,263,Центр,Стандарт,3.8,3.7,2.5,6,305,Нет,1
1,152,Центр,Экспресс,8.5,3.8,4.0,3,1186,Нет,0
2,241,Восток,Стандарт,44.7,43.5,4.7,1,529,Да,0
3,370,Центр,Бесконтактная,3.1,3.0,4.2,5,3216,Да,0
4,399,Юг,Стандарт,13.5,8.2,4.1,3,556,Да,0
...,...,...,...,...,...,...,...,...,...,...
495,319,Центр,Стандарт,1.7,1.0,3.9,7,210,Да,1
496,437,Центр,Стандарт,3.8,3.0,4.2,5,3800,Нет,0
497,235,Центр,Экспресс,6.9,1.9,4.1,1,2812,Нет,0
498,334,Центр,Стандарт,10.0,5.9,4.4,8,1378,Нет,0


## ДЗ. С пропусками, выбросами, и гипотезами

In [45]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# ========== ДАТАСЕТ 1: КОСМЕТОЛОГИЯ ==========

def generate_cosmetology_dataset(n=500, seed=42):
    np.random.seed(seed)
    data = {
        'client_age': np.random.randint(18, 70, n).astype(float),
        'procedure_price': np.random.normal(2500, 800, n).astype(float),
        'procedure_duration': np.random.normal(60, 20, n).astype(float),
        'master_rating': np.random.uniform(3.0, 5.0, n).astype(float),
        'return_visits': np.random.poisson(2, n).astype(float),
        'satisfaction_score': np.random.uniform(1, 10, n).astype(float)
    }
    df = pd.DataFrame(data)

    # Пропуски: MCAR
    mcar_idx = np.random.choice(df.index, size=int(n*0.05), replace=False)
    df.loc[mcar_idx, 'satisfaction_score'] = np.nan

    # Пропуски: MAR (зависят от возраста: старше 55 реже указывают длительность)
    mar_condition = df['client_age'] > 55
    mar_idx = df[mar_condition].sample(frac=0.15).index
    df.loc[mar_idx, 'procedure_duration'] = np.nan

    # Пропуски: MNAR (пропуск цены, когда цена аномально низкая)
    mnar_condition = df['procedure_price'] < 1200
    mnar_idx = df[mnar_condition].sample(frac=0.3).index
    df.loc[mnar_idx, 'procedure_price'] = np.nan

    # Выбросы: экстремально высокая цена (реальный выброс — ошибка ввода)
    outlier_idx_1 = np.random.choice(df.index, size=5, replace=False)
    df.loc[outlier_idx_1, 'procedure_price'] = np.random.uniform(50000, 70000, 5)

    # Выбросы: длительность процедуры 200+ (возможно, реальная, но редкая — какая-то сложная процедура)
    outlier_idx_2 = np.random.choice(df.index, size=8, replace=False)
    df.loc[outlier_idx_2, 'procedure_duration'] = np.random.uniform(200, 300, 8)

    return df

In [46]:
generate_cosmetology_dataset(n=500, seed=42)

,client_age,procedure_price,procedure_duration,master_rating,return_visits,satisfaction_score
0,56.0,4197.724958,40.482535,4.065916,2.0,3.623148
1,69.0,3325.972208,81.072836,3.648668,1.0,2.867868
2,46.0,1284.504027,41.012022,3.666004,1.0,3.143320
3,32.0,2112.612742,112.647641,4.338974,6.0,9.175911
4,60.0,3513.528919,69.866358,4.988279,1.0,NaN
...,...,...,...,...,...,...
495,65.0,1895.693654,44.600535,4.812287,2.0,6.703404
496,42.0,2010.785758,64.675718,3.553809,0.0,6.341152
497,57.0,1374.671123,28.882087,4.967043,4.0,1.146191
498,62.0,1761.413403,NaN,3.281423,2.0,7.559002


In [47]:
df = generate_cosmetology_dataset(n=500, seed=42)
df

,client_age,procedure_price,procedure_duration,master_rating,return_visits,satisfaction_score
0,56.0,4197.724958,40.482535,4.065916,2.0,3.623148
1,69.0,3325.972208,81.072836,3.648668,1.0,2.867868
2,46.0,1284.504027,41.012022,3.666004,1.0,3.143320
3,32.0,2112.612742,112.647641,4.338974,6.0,9.175911
4,60.0,3513.528919,69.866358,4.988279,1.0,NaN
...,...,...,...,...,...,...
495,65.0,1895.693654,44.600535,4.812287,2.0,6.703404
496,42.0,2010.785758,64.675718,3.553809,0.0,6.341152
497,57.0,1374.671123,28.882087,4.967043,4.0,1.146191
498,62.0,1761.413403,NaN,3.281423,2.0,7.559002


In [50]:
# df.to_csv('data/cosmetology.csv', index=False)

In [51]:
a = pd.read_csv('data/cosmetology.csv')
a

,client_age,procedure_price,procedure_duration,master_rating,return_visits,satisfaction_score
0,56.0,4197.724958,40.482535,4.065916,2.0,3.623148
1,69.0,3325.972208,81.072836,3.648668,1.0,2.867868
2,46.0,1284.504027,41.012022,3.666004,1.0,3.143320
3,32.0,2112.612742,112.647641,4.338974,6.0,9.175911
4,60.0,3513.528919,69.866358,4.988279,1.0,NaN
...,...,...,...,...,...,...
495,65.0,1895.693654,44.600535,4.812287,2.0,6.703404
496,42.0,2010.785758,64.675718,3.553809,0.0,6.341152
497,57.0,1374.671123,28.882087,4.967043,4.0,1.146191
498,62.0,1761.413403,NaN,3.281423,2.0,7.559002


In [52]:
# ========== ДАТАСЕТ 2: ЛОГИСТИКА ==========

def generate_logistics_dataset(n=500, seed=123):
    np.random.seed(seed)
    data = {
        'distance_km': np.random.exponential(15, n).astype(float),
        'weight_kg': np.random.uniform(0.5, 50, n).astype(float),
        'delivery_time_min': np.random.normal(45, 15, n).astype(float),
        'delivery_cost': np.random.normal(350, 100, n).astype(float),
        'courier_rating': np.random.uniform(2.0, 5.0, n).astype(float),
        'num_stops': np.random.poisson(3, n).astype(float)
    }
    df = pd.DataFrame(data)

    # Пропуски: MCAR — рейтинг курьера
    mcar_idx = np.random.choice(df.index, size=int(n*0.06), replace=False)
    df.loc[mcar_idx, 'courier_rating'] = np.nan

    # Пропуски: MAR — время доставки отсутствует, если расстояние > 30 км (курьер забыл отметить)
    mar_condition = df['distance_km'] > 30
    mar_idx = df[mar_condition].sample(frac=0.2).index
    df.loc[mar_idx, 'delivery_time_min'] = np.nan

    # Пропуски: MNAR — стоимость доставки пропущена, когда она аномально низкая (< 150)
    mnar_condition = df['delivery_cost'] < 150
    mnar_idx = df[mnar_condition].sample(frac=0.25).index
    df.loc[mnar_idx, 'delivery_cost'] = np.nan

    # Выбросы: экстремально большой вес (явный выброс — ошибка взвешивания)
    outlier_idx_1 = np.random.choice(df.index, size=4, replace=False)
    df.loc[outlier_idx_1, 'weight_kg'] = np.random.uniform(200, 500, 4)

    # Выбросы: очень малое время доставки при большом расстоянии (сомнительно — возможно, артефакт)
    outlier_idx_2 = np.random.choice(df.index, size=6, replace=False)
    df.loc[outlier_idx_2, 'delivery_time_min'] = np.random.uniform(1, 5, 6)

    return df

In [53]:
df = generate_logistics_dataset(n=500, seed=42)
df

,distance_km,weight_kg,delivery_time_min,delivery_cost,courier_rating,num_stops
0,7.039021,35.059005,47.665515,363.354090,4.914598,7.0
1,45.151821,27.036770,24.969835,334.753016,2.994041,2.0
2,19.751185,15.821617,50.702968,420.810868,3.446123,7.0
3,13.694138,40.782853,54.158786,445.670232,2.588293,4.0
4,2.544373,34.394193,53.396857,271.401054,NaN,3.0
...,...,...,...,...,...,...
495,6.539303,5.033313,47.125754,217.997749,4.530949,2.0
496,13.143656,45.907022,79.789943,288.823091,4.943578,3.0
497,1.213834,7.272522,50.899768,346.296320,4.379429,1.0
498,54.974402,47.536749,47.880737,307.069778,4.561355,2.0


In [54]:
# df.to_csv('data/logistics.csv', index=False)

In [55]:
a = pd.read_csv('data/logistics.csv')
a

,distance_km,weight_kg,delivery_time_min,delivery_cost,courier_rating,num_stops
0,7.039021,35.059005,47.665515,363.354090,4.914598,7.0
1,45.151821,27.036770,24.969835,334.753016,2.994041,2.0
2,19.751185,15.821617,50.702968,420.810868,3.446123,7.0
3,13.694138,40.782853,54.158786,445.670232,2.588293,4.0
4,2.544373,34.394193,53.396857,271.401054,NaN,3.0
...,...,...,...,...,...,...
495,6.539303,5.033313,47.125754,217.997749,4.530949,2.0
496,13.143656,45.907022,79.789943,288.823091,4.943578,3.0
497,1.213834,7.272522,50.899768,346.296320,4.379429,1.0
498,54.974402,47.536749,47.880737,307.069778,4.561355,2.0
